# EDF to Jupyter Explorer

Load an EDF file from a local path, inspect the recording, and convert manageable sections into formats that are convenient for notebook exploration.

The notebook keeps the full recording as an `mne.io.Raw` object and creates smaller `pandas`/`xarray` views for interactive work. This avoids accidentally turning a very large EDF into an enormous in-memory table.

## 1. Imports

Install missing packages in your environment if needed:

```bash
pip install mne pandas numpy matplotlib xarray ipywidgets
```

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
    import mne
except ImportError as exc:
    raise ImportError("Install MNE first: pip install mne") from exc

try:
    import xarray as xr
except ImportError:
    xr = None

pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 20)

mne.set_log_level("INFO")

## 2. Choose EDF Path

Set `EDF_PATH` to the file you want to explore. If you mounted the Harris SMB share and created a symlink such as `data/harris`, paths can point through that symlink.

In [2]:
EDF_PATH = Path("/home/volkan/repos/hypnose-eeg-analysis/data/rawdata/rawdata/sub-055_id-368/ses-024_date-20260518/ephys/sub-055_ses-024_recording-001.edf")

if not EDF_PATH.exists():
    raise FileNotFoundError(f"EDF file not found: {EDF_PATH.resolve()}")

EDF_PATH.resolve()

PosixPath('/mnt/harris/hypnose/hypnose_eeg/rawdata/sub-055_id-368/ses-024_date-20260518/ephys/sub-055_ses-024_recording-001.edf')

## 3. Load EDF as MNE Raw

`raw` is the best primary format for EEG exploration in Python because it preserves channel metadata, sampling rate, annotations, and plotting utilities.

Use `preload=False` first for large EDF files. You can crop and load smaller sections later.

In [3]:
raw = mne.io.read_raw_edf(
    EDF_PATH,
    preload=False,
    infer_types=True,
    verbose=True,
)

raw

Extracting EDF parameters from /home/volkan/repos/hypnose-eeg-analysis/data/rawdata/rawdata/sub-055_id-368/ses-024_date-20260518/ephys/sub-055_ses-024_recording-001.edf...
Channel 'EEG EEG1A-B' recognized as type EEG (renamed to 'EEG1A-B').
Channel 'EEG EEG2A-B' recognized as type EEG (renamed to 'EEG2A-B').
Channel 'EMG EMG' recognized as type EMG (renamed to 'EMG').
Setting channel info structure...
Creating raw.info structure...


<RawEDF | sub-055_ses-024_recording-001.edf, 3 x 41251840 (80570.0 s), ~7 KiB, data not loaded>

## 4. Recording Summary

In [4]:
summary = {
    "file": str(EDF_PATH),
    "n_channels": len(raw.ch_names),
    "sampling_rate_hz": raw.info["sfreq"],
    "n_samples": raw.n_times,
    "duration_seconds": raw.n_times / raw.info["sfreq"],
    "duration_hours": raw.n_times / raw.info["sfreq"] / 3600,
    "meas_date": raw.info.get("meas_date"),
}

pd.Series(summary)

file                /home/volkan/repos/hypnose-eeg-analysis/data/r...
n_channels                                                          3
sampling_rate_hz                                                512.0
n_samples                                                    41251840
duration_seconds                                              80570.0
duration_hours                                              22.380556
meas_date                                   2026-05-18 15:48:36+00:00
dtype: object

In [5]:
channel_table = pd.DataFrame({
    "channel": raw.ch_names,
    "type": raw.get_channel_types(),
})

channel_table

,channel,type
0,EEG1A-B,eeg
1,EEG2A-B,eeg
2,EMG,emg


## 5. Annotations and Events

EDF files may include sleep stages, markers, or other annotations. This cell converts annotations to a table when present.

In [6]:
annotations_df = pd.DataFrame({
    "onset_seconds": raw.annotations.onset,
    "duration_seconds": raw.annotations.duration,
    "description": raw.annotations.description,
})

annotations_df

,onset_seconds,duration_seconds,description
0,0.0,0.0,RECORD START
1,0.0,0.0,Started Recording
2,80564.6,0.0,Stopped Recording


## 6. Pick Channels and Time Window

Create a smaller loaded view for notebook exploration. Adjust `START_SECONDS`, `DURATION_SECONDS`, and `CHANNELS`.

In [ ]:
START_SECONDS = 0
DURATION_SECONDS = 60

# Use None for all channels, or provide names such as ["C3", "C4", "O1", "O2"].
CHANNELS = None

tmin = START_SECONDS
tmax = START_SECONDS + DURATION_SECONDS

raw_window = raw.copy().crop(tmin=tmin, tmax=tmax, include_tmax=False)

if CHANNELS is not None:
    raw_window.pick(CHANNELS)

raw_window.load_data()
raw_window

## 7. Convert Window to Pandas

Wide format is convenient for plotting and quick inspection: one row per time point, one column per channel.

In [ ]:
data, times = raw_window.get_data(return_times=True)

edf_wide_df = pd.DataFrame(data.T, columns=raw_window.ch_names)
edf_wide_df.insert(0, "time_seconds", times + START_SECONDS)

edf_wide_df.head()

In [ ]:
edf_long_df = edf_wide_df.melt(
    id_vars="time_seconds",
    var_name="channel",
    value_name="value",
)

edf_long_df.head()

## 8. Convert Window to Xarray

`xarray` is useful for labeled multidimensional data. The output has dimensions `channel` and `time_seconds`.

In [ ]:
if xr is None:
    print("xarray is not installed. Install with: pip install xarray")
else:
    edf_xr = xr.DataArray(
        data,
        dims=("channel", "time_seconds"),
        coords={
            "channel": raw_window.ch_names,
            "time_seconds": times + START_SECONDS,
        },
        attrs={
            "source_file": str(EDF_PATH),
            "sampling_rate_hz": raw_window.info["sfreq"],
            "units": "MNE default SI units, usually volts for EEG",
        },
        name="edf_signal",
    )
    display(edf_xr)

## 9. Quick Plots

In [ ]:
plot_channels = raw_window.ch_names[: min(8, len(raw_window.ch_names))]

fig, ax = plt.subplots(figsize=(14, 6))
offset = 0.0

for channel in plot_channels:
    values = edf_wide_df[channel].to_numpy()
    scale = np.nanstd(values) or 1.0
    ax.plot(edf_wide_df["time_seconds"], values / scale + offset, label=channel)
    offset += 4.0

ax.set_xlabel("Time (seconds)")
ax.set_ylabel("Scaled amplitude with channel offsets")
ax.set_title("EDF signal preview")
ax.legend(loc="upper right")
plt.show()

In [ ]:
# MNE's built-in browser is often the best interactive view.
# In Jupyter, this may open an interactive widget or a separate window depending on your environment.
raw_window.plot(
    duration=min(30, DURATION_SECONDS),
    n_channels=min(20, len(raw_window.ch_names)),
    scalings="auto",
)

## 10. Optional: Downsample for Faster Exploration

Downsampling is useful when plotting long windows or exporting compact notebook-friendly tables.

In [ ]:
TARGET_SFREQ = 100

raw_window_resampled = raw_window.copy().resample(TARGET_SFREQ)
resampled_data, resampled_times = raw_window_resampled.get_data(return_times=True)

edf_resampled_df = pd.DataFrame(resampled_data.T, columns=raw_window_resampled.ch_names)
edf_resampled_df.insert(0, "time_seconds", resampled_times + START_SECONDS)

edf_resampled_df.head()

## 11. Optional: Save Notebook-Friendly Outputs

Use these only for small windows or downsampled data. Saving full overnight EDF recordings as CSV can be very large.

In [ ]:
OUTPUT_DIR = Path("../data/processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

stem = EDF_PATH.stem

# CSV for quick inspection.
csv_path = OUTPUT_DIR / f"{stem}_window_{START_SECONDS}s_{DURATION_SECONDS}s_resampled.csv"
edf_resampled_df.to_csv(csv_path, index=False)

# FIF preserves MNE metadata better than CSV.
fif_path = OUTPUT_DIR / f"{stem}_window_{START_SECONDS}s_{DURATION_SECONDS}s_raw.fif"
raw_window.save(fif_path, overwrite=True)

csv_path, fif_path